# BBLF AI Selector v2: Optimisation & Simulation Tool

Round: Pre Tournament
 
Optimisation Goal: select the optimal starting 12 players prior to the start of the tournament 

# 0. Prerequistes

In [32]:
pip install mip==1.16rc0

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 25.1.1 -> 25.3
[notice] To update, run: python.exe -m pip install --upgrade pip


In [33]:
# 0. Prerequistes

import pandas as pd
import numpy as np
import os
import random
from mip import Model, xsum, maximize, BINARY 

os.getcwd()
directory = 'C:/Users/dilan/OneDrive/Documents/Data Science Projects/Repos/Big-Bash-Fantasy-AI-v2'
add_data_directory = 'C:/Users/dilan/OneDrive/Documents/Data Science Projects/Repos/Big-Bash-Fantasy-AI-v2/data/add_data_created/pre_tourny/'
over_data_directory = 'C:/Users/dilan/OneDrive/Documents/Data Science Projects/Repos/Big-Bash-Fantasy-AI-v2/data/add_data_created/overall'
py_data_directory = 'C:/Users/dilan/OneDrive/Documents/Data Science Projects/Repos/Big-Bash-Fantasy-AI-v2/data/python_datasets'
mock_data_dir = 'C:/Users/dilan/OneDrive/Documents/Data Science Projects/Repos/Big-Bash-Fantasy-AI-v2/data/mock_data/'

# 1. Data Extraction 

In [34]:
# 1. Data Extraction 
# Current Round & Optimal Number of Rounds
current_round = 1
opt_round = 2

# Pull in player price data csv file 
price_df = pd.read_csv(os.path.join(add_data_directory,'player_price_2026.csv'), low_memory=False)

# Player Role Flags
price_df["Wk_f"] = np.where((price_df["Role"] == "WK"), 1, 0)
price_df["Bat_f"] = np.where((price_df["Role"] == "WK") |(price_df["Role"] == "BAT") | (price_df["Role"] == "ALLR") , 1, 0)
price_df["Bowl_f"] = np.where((price_df["Role"] == "BOWL") | (price_df["Role"] == "ALLR") , 1, 0)
price_df = price_df[["Full_Name", "Price", "Team","Wk_f", "Bat_f", "Bowl_f", "Role", "Available", "In_Team"]].rename(columns = {"Full_Name": "Name"}) 

team_fix_df = pd.read_csv(os.path.join(over_data_directory,'team_loc_fixture.csv'), low_memory=False)
# team_fix_df = team_fix_df[team_fix_df.Round >= current_round].dropna()
# team_fix_df = team_fix_df[team_fix_df.Round <= current_round + opt_round - 1].dropna()

# Join team fixture to player price to create a row for each game
game_df = pd.merge(price_df , team_fix_df, left_on = ["Team"], right_on = ["Team"], how = "left")

# Pull player expected points csv file
# sim_pts_df = pd.read_csv(os.path.join(mock_data_dir,'simulated_mock_data.csv'), low_memory=False).drop(["Unnamed: 0", "player"], axis = 1)
sim_pts_df = pd.read_csv(os.path.join(mock_data_dir,'simulated_mock_data.csv'), low_memory=False)

# Join on expected points for each game
player_df_raw = pd.merge(game_df, sim_pts_df, left_on = ["Name", "Team", "Opposition", "Venue", "Home_f","Round"], right_on = ["Name", "Team", "Opposition", "Venue", "Home_f","Round"], how = "left")
player_df_raw["sim_points"] = np.where((player_df_raw["Role"] == "WK"), player_df_raw["sim_points"] + 10.51, player_df_raw["sim_points"] + 4.53)
player_df_raw["weight"] = 1

# Aggregate by player name (calculate total expected points for each player for x number of rounds)
player_df = player_df_raw.groupby(['Name', 'Price', "Team", "Wk_f", "Bat_f", "Bowl_f", "Role", "weight","Available", "In_Team"], as_index=False).agg(
sim_points=('sim_points',"sum"))

# Create dataframe for next round expected points (For captain selection)
player_df_next_rnd = player_df_raw[player_df_raw.Round == current_round].groupby(['Name', 'Price', "Team", "Wk_f", "Bat_f", "Bowl_f", "Role", "weight","Available", "In_Team"], as_index=False).agg(
next_rnd_sim_points=('sim_points',"sum"))
player_df_sim_points = player_df_next_rnd[["Name", "Team", "next_rnd_sim_points"]]


# 2. Optimisation Process (Pre Tournament Selection) - Starting 12 Players

Optimisation Objective: Maximise the number of expected fantasy points

Constraints: 
1. Number of players selected must be 12
2. Atleast 1 wicketkeeper
3. Atleast 6 batters
4. Atleast 5 bowlers
5. Total budget of team is less than $1,783,500 (As bench costs $216,500)

In [35]:
# Optimisation Variable Setup
points = player_df["sim_points"]
price = player_df["Price"]

weight = player_df["weight"]
available = player_df["Available"]
in_team = player_df["In_Team"]
wk_weight = player_df["Wk_f"]
bat_weight = player_df["Bat_f"]
bowl_weight = player_df["Bowl_f"]

play_cnt, total_player = 12, range(len(price))
# team_play_cnt, total_team_player = 0, range(len(price)) # IGNORE as pre tourny no players in team
wk_cnt, total_wk = 1, range(len(price))
bat_cnt, total_bat = 6, range(len(price))
bowl_cnt, total_bowl = 5, range(len(price))
budget, total_budget = 1783500, range(len(price))

# MIP Optimsation Setup
m = Model("knapsack")
x = [m.add_var(var_type=BINARY) for i in total_player]
print(x)

m.objective = maximize(xsum(points[i]*x[i] for i in total_player))

m += xsum(weight[i] * x[i] for i in total_player) == play_cnt
m += xsum(wk_weight[i] * x[i] for i in total_wk) >= wk_cnt
m += xsum(bat_weight[i] * x[i] for i in total_bat) >= bat_cnt
m += xsum(bowl_weight[i] * x[i] for i in total_bowl) >= bowl_cnt
m += xsum(price[i] * x[i] for i in total_budget) <= budget
m += xsum(available[i] * x[i] for i in total_player) == play_cnt
# m += xsum(in_team[i] * x[i] for i in total_team_player) >= team_play_cnt

# Optimisation Process & Results
m.optimize()
selected = [i for i in total_player if x[i].x >= 0.99]
print("selected items: {}".format(selected))

# Optimal Team Output
sel_player_df = player_df.iloc[selected]
sel_player_df = pd.merge(sel_player_df, player_df_next_rnd, left_on = ["Name","Team"], right_on = ["Name","Team"], how = "left")
print("Total Simulated Points:", sum(sel_player_df["sim_points"]))
print("Total Next Round Points:", sum(sel_player_df["next_rnd_sim_points"]))
print("Total Team Cost:", sum(sel_player_df["Price"]))
print("Number of Wk:", sum(sel_player_df["Wk_f"]))
print("Number of Bat:", sum(sel_player_df["Bat_f"]))
print("Number of Bowl:", sum(sel_player_df["Bowl_f"]))
print("Available Players:", sum(sel_player_df["Available"]))
print("Current Players Remaining:", sum(sel_player_df["In_Team"]))

print(sel_player_df.sort_values(by = "Price", ascending = False))
print(sel_player_df[sel_player_df['In_Team'] == 1].sort_values(by = "Price", ascending = False))
print(sel_player_df[sel_player_df['In_Team'] == 0].sort_values(by = "Price", ascending = False))

# Save Optimal Team to CSV
# sel_player_df.to_csv('C:/Users/dilan/OneDrive/Documents/Data Science Projects/Big Bash Fantasy AI/data/python_datasets/optimal_pre_tourny_team.csv')
sel_player_df.to_csv(os.path.join(add_data_directory,'optimal_pre_tourny_team.csv'))

[<mip.entities.Var object at 0x0000022AC731BD00>, <mip.entities.Var object at 0x0000022AC731AE00>, <mip.entities.Var object at 0x0000022AC7319180>, <mip.entities.Var object at 0x0000022AC731A050>, <mip.entities.Var object at 0x0000022AC731BF40>, <mip.entities.Var object at 0x0000022AC731ACE0>, <mip.entities.Var object at 0x0000022AC4C45600>, <mip.entities.Var object at 0x0000022AC4C44E20>, <mip.entities.Var object at 0x0000022AC4C45360>, <mip.entities.Var object at 0x0000022AC4C450F0>, <mip.entities.Var object at 0x0000022AC4C44C40>, <mip.entities.Var object at 0x0000022AC4C47DC0>, <mip.entities.Var object at 0x0000022AC4C47D60>, <mip.entities.Var object at 0x0000022AC4C45210>, <mip.entities.Var object at 0x0000022AC4C46470>, <mip.entities.Var object at 0x0000022AC4C44FA0>, <mip.entities.Var object at 0x0000022AC4C45E10>, <mip.entities.Var object at 0x0000022AC4C448E0>, <mip.entities.Var object at 0x0000022AC4C44C70>, <mip.entities.Var object at 0x0000022AC4C44370>, <mip.entities.Var o

KeyError: 'Price'